In [1]:
import json
import torch
import torch.nn as nn
import math
from huggingface_hub import login, hf_hub_download
from datasets import Dataset
from transformers import PreTrainedTokenizerFast, TrainingArguments, Trainer, DataCollatorForTokenClassification
from safetensors.torch import load_file


In [ ]:

login(token=os.environ["HF_TOKEN"])

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO = "A-240242/legal-gqa-checkpoints" 
HF_TOKENIZER_REPO= "A-240242/legal-modern-decoder-checkpoints"

In [3]:
def load_phase2_file(filename):
    path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=f"Phase_2_LNER/Train/{filename}", repo_type="dataset", token=HF_TOKEN)
    if filename.endswith(".txt"):
        with open(path, "r", encoding="utf-8") as f:
            return [line.strip() for line in f.readlines() if line.strip()]
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 2 Data...")
# Load and merge text dictionaries
text_data = load_phase2_file("fold1-text.json")
text_data.update(load_phase2_file("fold2-text.json"))

# Load and merge gold annotation dictionaries
gold_data = load_phase2_file("fold1-gold.json")
gold_data.update(load_phase2_file("fold2-gold.json"))

entity_types = load_phase2_file("labels.txt")
print(f"Loaded {len(text_data)} total documents across both folds.")

fold1-text.json:   0%|          | 0.00/1.11M [00:00<?, ?B/s]

fold2-text.json:   0%|          | 0.00/1.12M [00:00<?, ?B/s]

fold1-gold.json:   0%|          | 0.00/461k [00:00<?, ?B/s]

fold2-gold.json:   0%|          | 0.00/475k [00:00<?, ?B/s]

labels.txt:   0%|          | 0.00/71.0 [00:00<?, ?B/s]

Loaded 70 total documents across both folds.


In [4]:
# 1. Build BIO Schema
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])

LABEL2ID = {label: i for i, label in enumerate(LABEL_LIST)}
NUM_LABELS = len(LABEL_LIST)

# 2. Load Phase 1 Tokenizer
tokenizer = PreTrainedTokenizerFast.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"

# 3. Manual Sliding Window Alignment
def align_spans_to_tokens(raw_text, gold_spans, tokenizer, max_length=1024, stride=256):
    # Tokenize the ENTIRE document without truncation
    tokenized_full = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
    full_ids = tokenized_full["input_ids"]
    full_offsets = tokenized_full["offset_mapping"]
    
    chunked_data = {"input_ids": [], "attention_mask": [], "labels": []}
    step = max_length - stride
    
    # Manually slide a window across the massive token arrays
    for i in range(0, len(full_ids), step):
        chunk_ids = full_ids[i : i + max_length]
        chunk_offs = full_offsets[i : i + max_length]
        
        pad_len = max_length - len(chunk_ids)
        chunk_ids = chunk_ids + [tokenizer.pad_token_id] * pad_len
        chunk_offs = chunk_offs + [(0, 0)] * pad_len
        chunk_mask = [1] * (max_length - pad_len) + [0] * pad_len
        
        chunk_labels = []
        current_entity_id = None
        
        for tok_start, tok_end in chunk_offs:
            if tok_start == tok_end:
                chunk_labels.append(-100)
                continue

            sub_text = raw_text[tok_start:tok_end]
            clean_tok_start = tok_start + (len(sub_text) - len(sub_text.lstrip()))
            clean_tok_end = tok_end - (len(sub_text) - len(sub_text.rstrip()))

            matched_label, matched_span_idx = "O", None
            
            for s_idx, span in enumerate(gold_spans):
                g_start, g_end, g_label = span[:3] 
                if not (clean_tok_end <= g_start or clean_tok_start >= g_end):
                    matched_label = g_label
                    matched_span_idx = s_idx
                    break

            if matched_label == "O":
                chunk_labels.append(LABEL2ID["O"])
                current_entity_id = None
            else:
                if current_entity_id == matched_span_idx:
                    chunk_labels.append(LABEL2ID[f"I-{matched_label}"])
                else:
                    chunk_labels.append(LABEL2ID[f"B-{matched_label}"])
                    current_entity_id = matched_span_idx
                    
        chunked_data["input_ids"].append(chunk_ids)
        chunked_data["attention_mask"].append(chunk_mask)
        chunked_data["labels"].append(chunk_labels)

    return chunked_data

tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

In [5]:
print("Processing and Aligning Training Spans...")
all_features = {"input_ids": [], "attention_mask": [], "labels": []}

for doc_id, raw_text in text_data.items():
    gold_spans = gold_data.get(doc_id, [])
    aligned = align_spans_to_tokens(raw_text, gold_spans, tokenizer)
    
    all_features["input_ids"].extend(aligned["input_ids"])
    all_features["attention_mask"].extend(aligned["attention_mask"])
    all_features["labels"].extend(aligned["labels"])

train_dataset = Dataset.from_dict(all_features)
print(f"Successfully chunked into {len(train_dataset)} training sequences.")

Processing and Aligning Training Spans...
Successfully chunked into 636 training sequences.


In [6]:
import math
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import PretrainedConfig, Trainer, TrainingArguments, DataCollatorForTokenClassification
from transformers.modeling_outputs import TokenClassifierOutput
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION & LORMA ADAPTER
# ==========================================
class CustomLegalConfig(PretrainedConfig):
    model_type = "custom_legal_encdec"
    def __init__(self, vocab_size=32000, d_model=768, num_layers=6, num_heads=12, d_ff=2048, pad_token_id=0, eos_token_id=0, **kwargs):
        super().__init__(pad_token_id=pad_token_id, eos_token_id=eos_token_id, **kwargs)
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers
        self.num_heads = num_heads
        self.d_ff = d_ff

class LormaAdapter(nn.Module):
    def __init__(self, d_model, rank=128, alpha=128):
        super().__init__()
        self.scaling = alpha / rank
        self.A = nn.Linear(d_model, rank, bias=False)
        self.B = nn.Linear(rank, d_model, bias=False)
        nn.init.normal_(self.A.weight, std=1e-3)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.scaling * self.B(self.A(x))

# ==========================================
# 2. UNPACKED ENCODER & DECODER LAYERS
# ==========================================
class CustomLormaEncoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)

    def forward(self, src, src_key_padding_mask=None):
        src2, _ = self.self_attn(src, src, src, key_padding_mask=src_key_padding_mask)
        src2 = src2 + self.lorma_attn(src2)
        src = src + self.dropout1(src2)
        src = self.norm1(src)
        
        src2 = self.linear2(self.dropout(self.activation(self.linear1(src))))
        src2 = src2 + self.lorma_mlp(src2)
        src = src + self.dropout2(src2)
        src = self.norm2(src)
        return src

class CustomLormaDecoderLayer(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_self_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.multihead_attn = nn.MultiheadAttention(config.d_model, config.num_heads, dropout=0.1, batch_first=True)
        self.lorma_cross_attn = LormaAdapter(config.d_model, rank=rank)
        
        self.linear1 = nn.Linear(config.d_model, config.d_ff)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(0.1)
        self.linear2 = nn.Linear(config.d_ff, config.d_model)
        self.lorma_mlp = LormaAdapter(config.d_model, rank=rank)
        
        self.norm1 = nn.LayerNorm(config.d_model)
        self.norm2 = nn.LayerNorm(config.d_model)
        self.norm3 = nn.LayerNorm(config.d_model)
        self.dropout1 = nn.Dropout(0.1)
        self.dropout2 = nn.Dropout(0.1)
        self.dropout3 = nn.Dropout(0.1)

    def forward(self, tgt, memory, tgt_mask=None, memory_key_padding_mask=None, tgt_key_padding_mask=None):
        tgt2, _ = self.self_attn(tgt, tgt, tgt, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        tgt2 = tgt2 + self.lorma_self_attn(tgt2)
        tgt = tgt + self.dropout1(tgt2)
        tgt = self.norm1(tgt)
        
        tgt2, _ = self.multihead_attn(tgt, memory, memory, key_padding_mask=memory_key_padding_mask)
        tgt2 = tgt2 + self.lorma_cross_attn(tgt2)
        tgt = tgt + self.dropout2(tgt2)
        tgt = self.norm2(tgt)
        
        tgt2 = self.linear2(self.dropout(self.activation(self.linear1(tgt))))
        tgt2 = tgt2 + self.lorma_mlp(tgt2)
        tgt = tgt + self.dropout3(tgt2)
        tgt = self.norm3(tgt)
        return tgt

# ==========================================
# 3. FULL ENCODER-DECODER NER MODEL
# ==========================================
class LormaEncDecForNER(nn.Module):
    def __init__(self, config, rank=128):
        super().__init__()
        self.config = config
        self.d_model = config.d_model
        
        self.embedding = nn.Embedding(config.vocab_size, config.d_model, padding_idx=config.pad_token_id)
        # Match Phase 1 max sequence length (1024)
        self.pos_embedding = nn.Embedding(1024, config.d_model)
        
        self.encoder = nn.ModuleList([CustomLormaEncoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        self.decoder = nn.ModuleList([CustomLormaDecoderLayer(config, rank=rank) for _ in range(config.num_layers)])
        
        self.classifier = nn.Linear(config.d_model, NUM_LABELS)

    def _get_pos_ids(self, x):
        return torch.arange(x.size(1), device=x.device).unsqueeze(0).expand(x.size(0), -1)

    def forward(self, input_ids, attention_mask=None, labels=None, **kwargs):
        device = input_ids.device
        
        # Encoder Pass
        src_pos = self._get_pos_ids(input_ids)
        x = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(src_pos)
        src_key_padding_mask = (attention_mask == 0) if attention_mask is not None else None
        
        for layer in self.encoder:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        memory = x
        
        # Decoder Pass (Evaluating full sequence context)
        tgt_pos = self._get_pos_ids(input_ids)
        tgt = self.embedding(input_ids) * math.sqrt(self.d_model) + self.pos_embedding(tgt_pos)
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(input_ids.size(1), device=device)
        
        for layer in self.decoder:
            tgt = layer(
                tgt, 
                memory, 
                tgt_mask=tgt_mask, 
                memory_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=src_key_padding_mask
            )
            
        logits = self.classifier(tgt)
        
        loss = None
        if labels is not None:
            class_weights = torch.ones(logits.size(-1), device=device) * 3.0
            class_weights[0] = 1.0  
            loss_fn = nn.CrossEntropyLoss(weight=class_weights, ignore_index=-100)
            loss = loss_fn(logits.view(-1, logits.size(-1)), labels.view(-1))
            
        return TokenClassifierOutput(loss=loss, logits=logits)

# ==========================================
# 4. INITIALIZATION & WEIGHT MAPPING
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Initializing LORMA Encoder-Decoder Architecture...")
config = CustomLegalConfig(
    vocab_size=len(tokenizer), 
    pad_token_id=tokenizer.pad_token_id, 
    eos_token_id=tokenizer.eos_token_id
)
model = LormaEncDecForNER(config, rank=128)

print("Loading and Remapping Phase 1 Weights...")
HF_MODEL_REPO = "A-240242/legal-encdec-checkpoints"
model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO, filename="model.safetensors", token=HF_TOKEN)
state_dict = load_file(model_weight_path)

# Map original generic nn.Transformer attributes to our unrolled ModuleLists
mapped_dict = {}
for key, val in state_dict.items():
    if key.startswith("encoder.layers."):
        mapped_dict[key.replace("encoder.layers.", "encoder.")] = val
    elif key.startswith("decoder.layers."):
        mapped_dict[key.replace("decoder.layers.", "decoder.")] = val
    else:
        mapped_dict[key] = val

model.load_state_dict(mapped_dict, strict=False)

# Freeze base, Unfreeze LORMA matrices + classifier
for param in model.parameters(): 
    param.requires_grad = False

for layer in model.encoder:
    for param in layer.lorma_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for layer in model.decoder:
    for param in layer.lorma_self_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_cross_attn.parameters(): param.requires_grad = True
    for param in layer.lorma_mlp.parameters(): param.requires_grad = True

for param in model.classifier.parameters(): 
    param.requires_grad = True

model.to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. LORMA trainable parameters: {trainable_params:,}")

Initializing LORMA Encoder-Decoder Architecture...
Loading and Remapping Phase 1 Weights...


model.safetensors: reconstructing file:   0%|          |  0.00B /  521MB            

model.safetensors: downloading bytes:           |  0.00B            

Model ready. LORMA trainable parameters: 5,917,465


In [7]:
# ==========================================
# 5. TRAINING EXECUTION
# ==========================================
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir="./phase2_encdec_lorma",
    report_to="none",
    num_train_epochs=25, 
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-3, # Aggressive LR needed to wake up zero-initialized LORMA blocks         
    weight_decay=0.01,
    logging_steps=100,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=data_collator,
)

print("\nCommencing LORMA Encoder-Decoder Training for L-NER...")
trainer.train()


Commencing LORMA Encoder-Decoder Training for L-NER...


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
100,18.834569
200,17.510818
300,17.436150
400,17.373638
500,17.481692


/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(
/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/l

TrainOutput(global_step=500, training_loss=17.727373291015624, metrics={'train_runtime': 6043.9035, 'train_samples_per_second': 2.631, 'train_steps_per_second': 0.083, 'total_flos': 0.0, 'train_loss': 17.727373291015624, 'epoch': 25.0})

In [8]:
# 4. Inference Decoding Function with Relaxed Span Merger
def decode_token_predictions_to_spans(raw_text, chunks_preds, chunks_offsets):
    extracted_spans = []

    for preds, offsets in zip(chunks_preds, chunks_offsets):
        current_entity = None 

        for pred_id, (tok_start, tok_end) in zip(preds, offsets):
            if tok_start == tok_end:
                continue

            tag = ID2LABEL.get(pred_id, "O")

            if tag.startswith("B-"):
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                current_entity = {"label": tag[2:], "start": tok_start, "end": tok_end}

            elif tag.startswith("I-"):
                label_type = tag[2:]
                if current_entity is not None and current_entity["label"] == label_type:
                    current_entity["end"] = tok_end
                else:
                    if current_entity is not None:
                        extracted_spans.append(current_entity)
                    current_entity = {"label": label_type, "start": tok_start, "end": tok_end}
            else: 
                if current_entity is not None:
                    extracted_spans.append(current_entity)
                    current_entity = None

        if current_entity is not None:
            extracted_spans.append(current_entity)

   # --- THE FIX: Smart Gap Merger ---
    merged_spans = []
    for span in extracted_spans:
        if not merged_spans:
            merged_spans.append(span)
            continue
        
        prev = merged_spans[-1]
        gap = span["start"] - prev["end"]
        
        if span["label"] == prev["label"] and 0 <= gap <= 12:
            # Extract the exact text between the two fragments
            gap_text = raw_text[prev["end"]:span["start"]].lower().strip()
            
            # 1. Is the gap purely whitespace, punctuation, OR numbers (to catch missed months like ".10.")?
            is_safe_gap = all(c in ".,/- &0123456789" for c in gap_text)
            
            # 2. Is the gap a common legal connector?
            is_connector = gap_text in ["", "of", "v", "v.", "vs", "vs.", "and", "&"]
            
            if is_safe_gap or is_connector:
                prev["end"] = span["end"] 
                continue # Safely stitched!
            
        merged_spans.append(span)

    # Clean, Snap Boundaries, and Deduplicate
    final_spans = []
    seen = set()

    for span in merged_spans:
        s, e, label = span["start"], span["end"], span["label"]
        
        clean_start = s
        clean_end = e
        
        while clean_start < clean_end and (raw_text[clean_start].isspace() or raw_text[clean_start] in ".,;:-"):
            clean_start += 1
        while clean_end > clean_start and (raw_text[clean_end-1].isspace() or raw_text[clean_end-1] in ".,;:-"):
            clean_end -= 1

        if clean_start >= clean_end:
            continue

        span_tuple = (clean_start, clean_end, label)
        if span_tuple not in seen:
            seen.add(span_tuple)
            final_spans.append([clean_start, clean_end, label])

    final_spans.sort(key=lambda x: (x[0], x[1]))
    return final_spans

In [ ]:
# # --- Verification Test ---
# test_doc = "IN THE SUPREME COURT OF INDIA CRIMINAL APPELLATE JURISDICTION CRIMINAL APPEAL NO. 1128 OF 2026 RAMESH ... APPELLANT"
# test_gold = [
#     [7, 29, "COURT"],      # "SUPREME COURT OF INDIA"
#     [62, 98, "CASENO"],     # "CRIMINAL APPEAL NO. 1128 OF 2026"
#     [99, 105, "APP"]        # "RAMESH"
# ]

# # 1. Encode
# features = align_spans_to_tokens(test_doc, test_gold, tokenizer, max_length=128, stride=32)

# # 2. Simulate perfect model predictions by re-using gold label targets
# simulated_preds = [
#     [lbl if lbl != -100 else 0 for lbl in chunk_lbls] 
#     for chunk_lbls in features["labels"]
# ]

# # 3. Decode
# recovered_spans = decode_token_predictions_to_spans(
#     test_doc, 
#     features["input_ids"], 
#     features["offset_mapping"], 
#     simulated_preds
# )

# # 4. Assert Exact Character-Level Match
# print("Original Gold Spans: ", test_gold)
# print("Recovered Spans:     ", recovered_spans)

# assert recovered_spans == test_gold, "Alignment failed! Offsets do not match exactly."
# print("\nAlignment pipeline verified: 100% character-offset correctness achieved.")

In [9]:
# --- Cell 6: Bulletproof Inference & CodaBench Submission ---
import zipfile
import json
import torch
import gc
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
import os

print("Loading Test Data...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_2_LNER/Test/fold3-text.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print(f"Successfully loaded {len(test_data)} documents from the test fold.")

# Rebuild ID2LABEL locally
LABEL_LIST = ["O"]
for entity in entity_types:
    LABEL_LIST.extend([f"B-{entity}", f"I-{entity}"])
ID2LABEL = {i: label for i, label in enumerate(LABEL_LIST)}

model.eval()
model.to("cuda")
predictions = {}

chunk_size = 1024
stride = 256
step = chunk_size - stride

for doc_id, raw_text in tqdm(test_data.items(), desc="Generating L-NER Spans"):
    torch.cuda.empty_cache()
    gc.collect()
    
    try:
        # 1. Tokenize document without truncation
        tokenized = tokenizer(raw_text, return_offsets_mapping=True, truncation=False)
        full_ids = tokenized["input_ids"]
        full_offsets = tokenized["offset_mapping"]
        
        doc_preds = []
        doc_offsets = []
        
        # 2. Slide padded window across document
        for i in range(0, len(full_ids), step):
            raw_chunk_ids = full_ids[i : i + chunk_size]
            raw_chunk_offs = full_offsets[i : i + chunk_size]
            actual_len = len(raw_chunk_ids)
            
            # Pad to 1024 matching the training configuration
            pad_len = chunk_size - actual_len
            padded_ids = raw_chunk_ids + [tokenizer.pad_token_id] * pad_len
            mask = [1] * actual_len + [0] * pad_len
            
            input_tensor = torch.tensor([padded_ids], dtype=torch.long, device="cuda")
            attn_mask_tensor = torch.tensor([mask], dtype=torch.long, device="cuda")
            
            # 3. Model Prediction
            with torch.inference_mode():
                outputs = model(input_ids=input_tensor, attention_mask=attn_mask_tensor)
                logits = outputs.logits
                preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]
                
            # Slice off the padding before decoding
            doc_preds.append(preds[:actual_len])
            doc_offsets.append(raw_chunk_offs)
            
        # 4. Decode tokens back to character spans
        extracted_spans = decode_token_predictions_to_spans(raw_text, doc_preds, doc_offsets)
        predictions[doc_id] = extracted_spans

    except Exception as e:
        print(f"\nError processing document {doc_id}: {str(e)}")
        predictions[doc_id] = []

# 5. Save to Dedicated Output Directory
output_dir = "./phase2_encdec_lorma"
os.makedirs(output_dir, exist_ok=True)
PREDICTION_PATH = os.path.join(output_dir, "prediction.json")
SUBMISSION_ZIP = os.path.join(output_dir, "submission.zip")

with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print(f"\nFinal sanity check: Saved {len(predictions.keys())} document IDs to prediction.json")
print(f"L-NER Predictions complete! Submission saved to: {SUBMISSION_ZIP}")

Loading Test Data...


fold3-text.json:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

Successfully loaded 35 documents from the test fold.


Generating L-NER Spans:   0%|          | 0/35 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(



Final sanity check: Saved 35 document IDs to prediction.json
L-NER Predictions complete! Submission saved to: ./phase2_encdec_lorma/submission.zip


In [10]:
# Run this in a new cell to diagnose boundary artifacts
first_doc_id = list(predictions.keys())[0]
raw_doc_text = test_data[first_doc_id]

print(f"--- Diagnostic: Top 20 Predictions for Doc {first_doc_id} ---")
for span in predictions[first_doc_id][:20]:
    start, end, label = span
    extracted_text = raw_doc_text[start:end]
    # Using repr() to expose hidden trailing spaces or newlines (e.g. 'Court \n')
    print(f"[{label: <10}] : {repr(extracted_text)}")

--- Diagnostic: Top 20 Predictions for Doc 1128835 ---
